# 02 — Purchase-order process discovery and bottleneck maps

Flattens the OCEL to the Purchase Order (`PO`) object perspective and runs classical process discovery on it: dominant variants, a heuristics net, a BPMN model, and a performance-annotated map for bottleneck analysis.

## Load and flatten to the Purchase Order perspective

In [ ]:
import os
import pm4py

ocel_path = os.path.join("..", "data", "BPIC19.jsonocel")
ocel = pm4py.read_ocel(ocel_path)
print(f"Loaded OCEL: {len(ocel.events):,} events, {len(ocel.objects):,} objects")

# PM4Py's OC-* discovery algorithms work on the full OCEL; flattening to one
# object type lets us reuse the mature single-case-notion toolkit (heuristics
# nets, BPMN, performance DFGs) for a first, explainable pass.
flat_po_log = pm4py.ocel_flattening(ocel, "PO")
print(f"Flattened log: {len(flat_po_log)} events")


## Dominant procurement paths

In [ ]:
variants = pm4py.get_variants(flat_po_log)
sorted_variants = sorted(variants.items(), key=lambda x: x[1], reverse=True)

print("Top 3 purchase-order variants by volume:")
for variant, count in sorted_variants[:3]:
    steps = " -> ".join(variant) if isinstance(variant, (tuple, list)) else str(variant)
    print(f"\n{count:,} purchase orders")
    print(steps)


## Heuristics net (frequency-filtered process map)

In [ ]:
os.makedirs("outputs", exist_ok=True)

heu_net = pm4py.discover_heuristics_net(flat_po_log)
pm4py.save_vis_heuristics_net(heu_net, "outputs/po_heuristics_net.png")
print("Saved outputs/po_heuristics_net.png")


## BPMN model

In [ ]:
bpmn_model = pm4py.discover_bpmn_inductive(flat_po_log)
pm4py.save_vis_bpmn(bpmn_model, "outputs/po_bpmn_model.png")
print("Saved outputs/po_bpmn_model.png")


## Performance map (bottleneck view)

**Note:** the unfiltered performance DFG over all 76k purchase orders renders as a dense, hard-to-read graph — that is the current state of `outputs/po_performance_map.png`. Filter to the top N variants first (e.g. `pm4py.filter_variants_top_k(flat_po_log, 10)`) so the saved map is actually legible before it goes in the README or the demo video.

In [ ]:
dfg, start_activities, end_activities = pm4py.discover_performance_dfg(flat_po_log)
pm4py.save_vis_performance_dfg(dfg, start_activities, end_activities, "outputs/po_performance_map.png")
print("Saved outputs/po_performance_map.png")


## Next steps (Week 2)

- Conformance-check every trace against the standard 3-way-match-with-GR path; quantify the overall deviation rate.
- Quantify the five findings: rework loops, price/quantity changes after goods receipt, duplicate/blocked invoices, days-to-clear spread across the 60 subsidiaries, and maverick buying.
- Replace `po_performance_map.png` with a top-variant-filtered version once it is legible at a glance.